In [ ]:
"""
Student Marks Analyzer (with user input) - Python + NumPy + Pandas
Enter students and marks, then get average, topper, grades, pass/fail.
"""
import numpy as np
import pandas as pd

PASS_MARKS = 33


def ask_int(prompt, low=None, high=None):
    while True:
        try:
            v = int(input(prompt))
            if (low is not None and v < low) or (high is not None and v > high):
                print(f"  Enter a number between {low} and {high}")
                continue
            return v
        except ValueError:
            print("  Please enter a valid number")


def ask_float(prompt, low=0, high=100):
    while True:
        try:
            v = float(input(prompt))
            if v < low or v > high:
                print(f"  Marks must be between {low} and {high}")
                continue
            return v
        except ValueError:
            print("  Please enter a valid number")


def get_data():
    n = ask_int("How many students? ", 1, 200)
    k = ask_int("How many subjects? ", 1, 10)
    subjects = [input(f"Subject {i + 1} name: ").strip().title() or f"Sub{i + 1}" for i in range(k)]

    names, marks = [], []
    for i in range(n):
        name = input(f"\nStudent {i + 1} name: ").strip().title() or f"Student{i + 1}"
        row = [ask_float(f"  {s} marks (0-100): ") for s in subjects]
        names.append(name)
        marks.append(row)
    return pd.DataFrame(marks, index=names, columns=subjects)


def grade(p):
    return ("A+" if p >= 90 else "A" if p >= 80 else "B" if p >= 70 else
            "C" if p >= 60 else "D" if p >= 50 else "E" if p >= PASS_MARKS else "F")


def analyze(df):
    res = df.copy()
    res["Total"] = df.sum(axis=1)
    res["Percent"] = np.round(df.mean(axis=1), 2)
    res["Grade"] = res["Percent"].apply(grade)
    res["Result"] = np.where((df >= PASS_MARKS).all(axis=1), "PASS", "FAIL")
    res["Rank"] = res["Total"].rank(ascending=False, method="min").astype(int)
    res = res.sort_values("Rank")

    print("\n" + "=" * 60)
    print("                 STUDENT RESULT SHEET")
    print("=" * 60)
    print(res.to_string())

    print("\n--- Class Summary ---")
    print(f"Class average   : {res.Percent.mean():.2f}%")
    print(f"Topper          : {res.index[0]} ({res.Percent.iloc[0]}%)")
    print(f"Lowest          : {res.index[-1]} ({res.Percent.iloc[-1]}%)")
    pass_n = (res.Result == "PASS").sum()
    print(f"Passed / Failed : {pass_n} / {len(res) - pass_n}")
    print(f"Pass percentage : {pass_n / len(res) * 100:.1f}%")

    print("\n--- Subject-wise ---")
    sub = pd.DataFrame({"Average": df.mean().round(1), "Highest": df.max(),
                        "Lowest": df.min(), "Topper": df.idxmax()})
    print(sub.to_string())

    print("\n--- Grade distribution ---")
    print(res.Grade.value_counts().sort_index().to_string())

    res.to_csv("student_results.csv")
    print("\nSaved: student_results.csv")


if __name__ == "__main__":
    print("=== STUDENT MARKS ANALYZER ===\n")
    analyze(get_data())

=== STUDENT MARKS ANALYZER ===

